# 57 — Ghép dữ liệu: merge · join · concat

Ghép là phép dễ viết nhất và dễ sai nhất trong pandas. Nó sai theo hai cách
**đều im lặng**: mất dòng, hoặc nhân dòng lên.

Notebook này dạy cách bắt cả hai *trước khi* chúng đi vào kết quả:

1. Bốn kiểu join, và cái nào mất dòng
2. ⚠️ **`validate=`** — tham số quan trọng nhất và ít người dùng nhất
3. **`indicator=`** — biết dòng nào từ đâu tới
4. `concat` và ⚠️ **`df.attrs` bốc hơi**
5. `merge_asof` — ghép theo thời gian gần nhất, cho dữ liệu tick

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import hom_nay, lui_ngay

client = finlens.client()
HOM_NAY = hom_nay(client)

danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")
MA = danh_muc["symbol"].tolist()[:80]
gia = client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, thang=6)).sort_values(["symbol", "date"])

print(f"pandas {pd.__version__} · giá {gia.shape} · danh mục {danh_muc.shape}")

pandas 3.0.5 · giá (9488, 7) · danh mục (407, 10)


## 1 · Bốn kiểu join

`how=` quyết định dòng nào sống sót. Đây là bảng đáng dán lên tường.

| `how` | Giữ dòng của | Mất dòng khi |
|---|---|---|
| `"inner"` (mặc định) | **cả hai** bên đều khớp | một bên thiếu khoá → **mất im lặng** |
| `"left"` | tất cả bên trái | không mất, bên phải thiếu thành `NaN` |
| `"right"` | tất cả bên phải | không mất, bên trái thiếu thành `NaN` |
| `"outer"` | **tất cả** cả hai bên | không mất gì |

In [2]:
# Bảng tra ngành, nhưng CỐ Ý thiếu vài mã để thấy hậu quả
nganh = danh_muc[["symbol", "icb_name2", "short_name"]].head(60)
print(f"Frame giá có {gia['symbol'].nunique()} mã")
print(f"Bảng ngành có {nganh['symbol'].nunique()} mã   ← thiếu {gia['symbol'].nunique() - nganh['symbol'].nunique()} mã")

ket = {}
for how in ("inner", "left", "right", "outer"):
    m = gia.merge(nganh, on="symbol", how=how)
    ket[how] = {"dòng": len(m), "mã": m["symbol"].nunique(), "ô NaN": int(m.isna().sum().sum())}

pd.DataFrame(ket).T

Frame giá có 79 mã
Bảng ngành có 60 mã   ← thiếu 19 mã


,dòng,mã,ô NaN
inner,7068,59,0
left,9488,79,4840
right,7069,60,6
outer,9489,80,4846


⚠️ Nhìn dòng `inner`: nó **mất hơn hai vạn dòng** so với `left`, và không có
cảnh báo nào. Đây là cách hỏng phổ biến nhất — bạn ghép thêm một bảng tra để
*làm giàu* dữ liệu, nhưng vô tình *lọc* nó.

In [3]:
mat = len(gia.merge(nganh, on="symbol", how="left")) - len(gia.merge(nganh, on="symbol"))
print(f"merge mặc định (inner) mất {mat:,} dòng = {mat / len(gia):.0%} dữ liệu")
print()
print("Các mã bị rơi:")
roi = sorted(set(gia["symbol"]) - set(nganh["symbol"]))
print(f"  {roi}")

merge mặc định (inner) mất 2,420 dòng = 26% dữ liệu

Các mã bị rơi:
  ['CMX', 'CNG', 'COM', 'CRC', 'CRE', 'CRV', 'CSM', 'CSV', 'CTD', 'CTF', 'CTG', 'CTI', 'CTR', 'CTS', 'CVT', 'D2D', 'DAH', 'DAT', 'DBC', 'DBD']


**Quy tắc:** khi ghép để *làm giàu* — thêm cột mô tả vào một bảng sự kiện —
hãy dùng `how="left"` và **kiểm số dòng sau khi ghép**.

In [4]:
truoc = len(gia)
lam_giau = gia.merge(nganh, on="symbol", how="left")
assert len(lam_giau) == truoc, f"Số dòng đổi: {truoc} → {len(lam_giau)}"
print(f"✓ Giữ nguyên {truoc:,} dòng")
print(f"  {lam_giau['icb_name2'].isna().sum():,} dòng không tra được ngành → NaN, không bị xoá")

✓ Giữ nguyên 9,488 dòng
  2,420 dòng không tra được ngành → NaN, không bị xoá


## 2 · ⚠️ `validate=` — tham số quan trọng nhất

Cách hỏng thứ hai ngược lại: **merge nhân dòng lên**. Nếu bên phải có khoá
trùng, mỗi dòng bên trái sẽ khớp với nhiều dòng bên phải.

`validate=` khai báo quan hệ bạn *tin* là đúng, và pandas ném lỗi nếu sai.

In [5]:
# Bảng tra bị trùng — chuyện rất hay xảy ra sau một lần append nhầm
nganh_trung = pd.concat([nganh, nganh.head(5)])

khong_kiem = gia.merge(nganh_trung, on="symbol", how="left")
print(f"Không validate → {len(gia):,} dòng thành {len(khong_kiem):,} dòng")
print(f"  phình thêm {len(khong_kiem) - len(gia):,} dòng, không cảnh báo nào")

try:
    gia.merge(nganh_trung, on="symbol", how="left", validate="many_to_one")
except pd.errors.MergeError as e:
    print(f"\nCó validate → MergeError: {e}")

Không validate → 9,488 dòng thành 10,030 dòng
  phình thêm 542 dòng, không cảnh báo nào

Có validate → MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 symbol
   AAA
   AAM
   AAN
   AAT
   ABR ...


Bốn giá trị của `validate=`:

| Giá trị | Khẳng định |
|---|---|
| `"one_to_one"` / `"1:1"` | khoá **duy nhất ở cả hai** bên |
| `"one_to_many"` / `"1:m"` | khoá duy nhất **bên trái** |
| `"many_to_one"` / `"m:1"` | khoá duy nhất **bên phải** — dùng cho bảng tra |
| `"many_to_many"` / `"m:m"` | không khẳng định gì (mặc định) |

**`"many_to_one"` là giá trị bạn cần trong 90% trường hợp**: nhiều dòng sự kiện
ghép với một bảng tra. Viết nó vào là bạn được bảo vệ miễn phí.

In [6]:
dung = gia.merge(nganh, on="symbol", how="left", validate="many_to_one")
print(f"✓ validate='many_to_one' qua được với bảng tra sạch: {len(dung):,} dòng")

✓ validate='many_to_one' qua được với bảng tra sạch: 9,488 dòng


## 3 · `indicator=` — dòng nào từ đâu tới

Khi bạn *muốn* biết bên nào thiếu thay vì chỉ đếm.

In [7]:
co_nhan = gia.merge(nganh, on="symbol", how="outer", indicator=True)
print(co_nhan["_merge"].value_counts().to_string())
print()
print("Các mã chỉ có ở bảng giá, không tra được ngành:")
chi_trai = co_nhan[co_nhan["_merge"] == "left_only"]["symbol"].unique()
print(f"  {sorted(chi_trai)}")

_merge
both          7068
left_only     2420
right_only       1

Các mã chỉ có ở bảng giá, không tra được ngành:
  ['CMX', 'CNG', 'COM', 'CRC', 'CRE', 'CRV', 'CSM', 'CSV', 'CTD', 'CTF', 'CTG', 'CTI', 'CTR', 'CTS', 'CVT', 'D2D', 'DAH', 'DAT', 'DBC', 'DBD']


## 4 · Ghép trên nhiều khoá

Dữ liệu thị trường hầu như luôn có khoá kép `(symbol, date)`.

In [8]:
dong_tien = client.eod.stock.investor.flow(MA, group="foreign", start=lui_ngay(HOM_NAY, thang=6))
print(f"Dòng tiền: {dong_tien.shape}")

ghep_kep = gia.merge(
    dong_tien[["symbol", "date", "net_value"]],
    on=["symbol", "date"],
    how="left",
    validate="one_to_one",
)
print(f"Ghép trên (symbol, date): {len(gia):,} → {len(ghep_kep):,} dòng")
print(f"  {ghep_kep['net_value'].isna().sum():,} dòng không có số liệu dòng tiền")

Dòng tiền: (9303, 9)
Ghép trên (symbol, date): 9,488 → 9,488 dòng
  185 dòng không có số liệu dòng tiền


⚠️ **Quên một khoá là thảm hoạ.** Ghép chỉ trên `symbol` sẽ khớp mỗi phiên giá
với *mọi* phiên dòng tiền của cùng mã đó:

In [9]:
n_phien = dong_tien.groupby("symbol", observed=True).size().median()
print(f"Mỗi mã có khoảng {n_phien:.0f} phiên dòng tiền")
print(f"Ghép đúng   (symbol, date) → {len(ghep_kep):,} dòng")
print(f"Ghép thiếu  chỉ 'symbol'   → ước tính {len(gia) * n_phien:,.0f} dòng "
      f"({n_phien:.0f} lần)")
print()
print("→ Đây là lúc validate='one_to_one' cứu bạn:")
try:
    gia.head(1000).merge(dong_tien, on="symbol", validate="one_to_one")
except pd.errors.MergeError as e:
    print(f"   MergeError: {str(e)[:100]}")

Mỗi mã có khoảng 120 phiên dòng tiền
Ghép đúng   (symbol, date) → 9,488 dòng
Ghép thiếu  chỉ 'symbol'   → ước tính 1,138,560 dòng (120 lần)

→ Đây là lúc validate='one_to_one' cứu bạn:
   MergeError: Merge keys are not unique in either left or right dataset; not a one-to-one merge.
Duplicates in lef


### `suffixes` khi trùng tên cột

In [10]:
# Xung đột thật: ghép giá đã điều chỉnh với giá thô — hai frame cùng bộ tên cột
tho = client.eod.stock.ohlcv(MA[:5], start=lui_ngay(HOM_NAY, thang=1), adjusted=False)
dieu_chinh = client.eod.stock.ohlcv(MA[:5], start=lui_ngay(HOM_NAY, thang=1))

mac_dinh = dieu_chinh.merge(tho, on=["symbol", "date"])
print("Không nêu suffixes — pandas tự thêm _x và _y:")
print(f"  {[c for c in mac_dinh.columns if c.startswith('close')]}")
print()
ro_rang = dieu_chinh.merge(tho, on=["symbol", "date"], suffixes=("_dc", "_tho"))
print("Nêu rõ suffixes:")
print(f"  {[c for c in ro_rang.columns if c.startswith('close')]}")
print()
print(f"Ví dụ một dòng: close_dc={ro_rang['close_dc'].iloc[0]:.2f} · close_tho={ro_rang['close_tho'].iloc[0]:.2f}")

Không nêu suffixes — pandas tự thêm _x và _y:
  ['close_x', 'close_y']

Nêu rõ suffixes:
  ['close_dc', 'close_tho']

Ví dụ một dòng: close_dc=6.93 · close_tho=7.23


⚠️ `_x` và `_y` là mặc định, và chúng **không nói gì cả**. Sáu tháng sau bạn sẽ
không nhớ `close_x` là giá điều chỉnh hay giá thô. Luôn nêu `suffixes=`.

## 5 · `concat` — xếp chồng, không ghép theo khoá

`merge` ghép theo **giá trị khoá**. `concat` chỉ **xếp chồng** theo trục.

In [11]:
a = gia[gia["symbol"] == MA[0]]
b = gia[gia["symbol"] == MA[1]]

doc = pd.concat([a, b])  # axis=0, mặc định
print(f"concat dọc  : {a.shape} + {b.shape} → {doc.shape}")

ngang = pd.concat([a.set_index("date")["close"], b.set_index("date")["close"]], axis=1)
ngang.columns = [MA[0], MA[1]]
print(f"concat ngang: → {ngang.shape}  (căn theo index)")

concat dọc  : (121, 7) + (121, 7) → (242, 7)
concat ngang: → (121, 2)  (căn theo index)


⚠️ **`concat` dọc giữ nguyên index cũ**, nên bạn được một frame có nhãn trùng
lặp — và mọi phép `.loc` sau đó trả về nhiều dòng hơn bạn tưởng.

In [12]:
# a và b tách từ cùng một frame nên index của chúng RỜI nhau — chưa thấy vấn đề
print(f"a và b tách từ một frame → index trùng sau concat: {doc.index.has_duplicates}")

# Nhưng hai frame lấy từ HAI lời gọi API khác nhau thì đều bắt đầu từ 0
a2 = client.eod.stock.ohlcv(MA[0], start=lui_ngay(HOM_NAY, thang=1))
b2 = client.eod.stock.ohlcv(MA[1], start=lui_ngay(HOM_NAY, thang=1))
print(f"Hai lời gọi riêng → index a2 bắt đầu {a2.index[0]}, b2 bắt đầu {b2.index[0]}")

doc2 = pd.concat([a2, b2])
print(f"concat → index trùng: {doc2.index.has_duplicates}")
nhan = doc2.index[0]
print(f"  doc2.loc[[{nhan}]] trả về {len(doc2.loc[[nhan]])} dòng — hai mã khác nhau!")
print(doc2.loc[[nhan], ["symbol", "date", "close"]].to_string())
print()
sach = pd.concat([a2, b2], ignore_index=True)
print(f"concat(ignore_index=True) → index trùng: {sach.index.has_duplicates}")

a và b tách từ một frame → index trùng sau concat: False


Hai lời gọi riêng → index a2 bắt đầu 0, b2 bắt đầu 0
concat → index trùng: True
  doc2.loc[[0]] trả về 2 dòng — hai mã khác nhau!
  symbol       date  close
0    AAA 2026-07-13   6.93
0    AAM 2026-07-13   6.33

concat(ignore_index=True) → index trùng: False


### ⚠️ `df.attrs` bốc hơi sau `concat`

Đây là cạm bẫy đã đo ở notebook `00`, và nó thuộc về chương này.

In [13]:
hpg = client.eod.stock.ohlcv("HPG", start=lui_ngay(HOM_NAY, thang=1))
cw = client.eod.warrant.ohlcv("CHPG2525", start=lui_ngay(HOM_NAY, thang=1))

print(f"HPG      close: {hpg['close'].iloc[-1]:>8,.1f}  đơn vị {hpg.attrs['finlens']['units']['close']}")
print(f"CHPG2525 close: {cw['close'].iloc[-1]:>8,.1f}  đơn vị {cw.attrs['finlens']['units']['close']}")

ghep = pd.concat([hpg, cw])
print(f"\nSau concat, attrs = {ghep.attrs}")
print(f"Cột close giờ chứa: {ghep.groupby('symbol', observed=True)['close'].last().tolist()}")
print("→ hai đơn vị lệch nhau 1000 lần, nằm cùng một cột, không dấu vết nào")

HPG      close:     22.2  đơn vị kVND
CHPG2525 close:  1,650.0  đơn vị VND

Sau concat, attrs = {}
Cột close giờ chứa: [1650.0, 22.2]
→ hai đơn vị lệch nhau 1000 lần, nằm cùng một cột, không dấu vết nào


**Vì sao pandas làm thế:** hai frame khai `units` khác nhau, nên `attrs` gộp
lại sẽ là một lời nói dối. pandas chọn bỏ trống thay vì đoán — đúng, nhưng im
lặng.

⚠️ Chú ý: nếu hai frame khai `attrs` **giống hệt nhau** thì pandas **giữ lại**.
Nên bạn không thể dựa vào "attrs luôn mất" để phát hiện lỗi:

In [14]:
hai_ma = client.eod.stock.ohlcv(["HPG", "VCB"], start=lui_ngay(HOM_NAY, thang=1))
p1 = hai_ma[hai_ma["symbol"] == "HPG"]
p2 = hai_ma[hai_ma["symbol"] == "VCB"]
giu = pd.concat([p1, p2])
print(f"concat hai phần cùng đơn vị → attrs còn: {'finlens' in giu.attrs}")
print(f"concat hai frame khác đơn vị → attrs còn: {'finlens' in ghep.attrs}")
print()
print("→ attrs mất hay không phụ thuộc DỮ LIỆU, không phải phép toán.")
print("  Đọc đơn vị vào biến TRƯỚC khi ghép là cách duy nhất chắc chắn.")

concat hai phần cùng đơn vị → attrs còn: True
concat hai frame khác đơn vị → attrs còn: False

→ attrs mất hay không phụ thuộc DỮ LIỆU, không phải phép toán.
  Đọc đơn vị vào biến TRƯỚC khi ghép là cách duy nhất chắc chắn.


## 6 · `merge_asof` — ghép theo thời gian gần nhất

Dùng khi hai chuỗi thời gian **không trùng mốc**: gán mỗi lệnh khớp vào thanh
giá 15 phút gần nhất, gán giá cuối ngày vào một sự kiện xảy ra giữa phiên.

`merge` thường sẽ không khớp gì cả vì mốc thời gian không bằng nhau tuyệt đối.

In [15]:
# ⚠️ Lấy PHIÊN GIAO DỊCH thật từ dữ liệu, đừng trừ ngày lịch — trừ 4 ngày từ
# thứ Tư là rơi vào thứ Bảy, và tick của ngày nghỉ là một frame rỗng.
NGAY = gia["date"].drop_duplicates().nlargest(3).iloc[-1].strftime("%Y-%m-%d")
tick = client.intraday.stock.ticks("HPG", date=NGAY)
thanh = client.intraday.stock.ohlcv("HPG", interval="15min", start=NGAY, end=NGAY)

print(f"Ngày {NGAY}: {len(tick):,} lệnh khớp · {len(thanh)} thanh 15 phút")
print(f"Mốc tick đầu : {tick['time'].iloc[0]}")
print(f"Mốc thanh đầu: {thanh['time'].iloc[0]}")

Ngày 2026-08-10: 3,501 lệnh khớp · 16 thanh 15 phút
Mốc tick đầu : 2026-08-10 09:15:15+07:00
Mốc thanh đầu: 2026-08-10 09:15:00+07:00


In [16]:
# merge thường: gần như không khớp gì
thuong = tick.merge(thanh[["time", "close"]], on="time", how="inner")
print(f"merge trên 'time' chính xác → {len(thuong)} dòng khớp / {len(tick):,} lệnh")

merge trên 'time' chính xác → 1 dòng khớp / 3,501 lệnh


In [17]:
# merge_asof: mỗi lệnh khớp gán vào thanh 15 phút GẦN NHẤT VỀ TRƯỚC
gan_nhat = pd.merge_asof(
    tick.sort_values("time"),
    thanh[["time", "close"]].sort_values("time").rename(columns={"close": "gia_thanh"}),
    on="time",
    direction="backward",
)
print(f"merge_asof → {gan_nhat['gia_thanh'].notna().sum():,}/{len(gan_nhat):,} lệnh gán được thanh")
gan_nhat[["time", "price", "volume", "side", "gia_thanh"]].head(5)

merge_asof → 3,501/3,501 lệnh gán được thanh


,time,price,volume,side,gia_thanh
0,2026-08-10 09:15:15+07:00,22.30,190900.0,auction,22.05
1,2026-08-10 09:15:29+07:00,22.25,3000.0,auction,22.05
2,2026-08-10 09:15:33+07:00,22.25,200.0,auction,22.05
3,2026-08-10 09:15:37+07:00,22.25,200.0,auction,22.05
4,2026-08-10 09:15:42+07:00,22.25,200.0,auction,22.05


⚠️ **`merge_asof` bắt buộc cả hai frame phải sắp xếp theo cột khoá.** Nếu chưa
sắp, nó ném lỗi — đây là một trong ít chỗ pandas không im lặng:

In [18]:
try:
    pd.merge_asof(
        tick.sort_values("price"),  # sắp sai cột
        thanh[["time", "close"]].sort_values("time"),
        on="time",
    )
except ValueError as e:
    print(f"Chưa sắp theo khoá → ValueError: {str(e)[:80]}")

Chưa sắp theo khoá → ValueError: left keys must be sorted


### `direction=` quyết định ý nghĩa

| `direction` | Lấy thanh | Dùng khi |
|---|---|---|
| `"backward"` (mặc định) | gần nhất **trước hoặc bằng** | gán giá đang có tại thời điểm đó |
| `"forward"` | gần nhất **sau hoặc bằng** | gán giá sẽ khớp tiếp theo |
| `"nearest"` | gần nhất về mọi phía | phân tích sau sự kiện, **có nhìn trước** |

⚠️ **`"nearest"` và `"forward"` nhìn vào tương lai.** Trong backtest chúng là
nguồn thiên lệch nhìn trước — dùng `"backward"` trừ khi bạn biết chắc mình đang
làm gì.

In [19]:
ba_huong = {}
for huong in ("backward", "forward", "nearest"):
    ba_huong[huong] = pd.merge_asof(
        tick.sort_values("time"),
        thanh[["time", "close"]].sort_values("time"),
        on="time",
        direction=huong,
    )["close"]

so = pd.DataFrame(ba_huong)
khac = (so.nunique(axis=1) > 1).sum()
print(f"{khac:,}/{len(so):,} lệnh khớp được gán giá KHÁC NHAU tuỳ direction")
print()
print("Vài lệnh nằm giữa hai thanh — ba hướng cho ba kết quả:")
vi_tri = so[so.nunique(axis=1) > 1].index[:4]
print(
    tick.loc[vi_tri, ["time", "price"]]
    .join(so.loc[vi_tri])
    .to_string(index=False)
)

1,877/3,501 lệnh khớp được gán giá KHÁC NHAU tuỳ direction

Vài lệnh nằm giữa hai thanh — ba hướng cho ba kết quả:


                     time  price  backward  forward  nearest
2026-08-10 09:30:04+07:00  22.05     22.05     22.1    22.05
2026-08-10 09:30:05+07:00  22.05     22.05     22.1    22.05
2026-08-10 09:30:07+07:00  22.05     22.05     22.1    22.05
2026-08-10 09:30:10+07:00  22.10     22.05     22.1    22.05

### `tolerance=` — từ chối ghép khi quá xa

In [20]:
co_han = pd.merge_asof(
    tick.sort_values("time"),
    thanh[["time", "close"]].sort_values("time"),
    on="time",
    direction="backward",
    tolerance=pd.Timedelta("5min"),
)
print(f"Không giới hạn: {gan_nhat['gia_thanh'].notna().sum():,} lệnh gán được")
print(f"tolerance=5min: {co_han['close'].notna().sum():,} lệnh gán được")
print(f"→ {co_han['close'].isna().sum():,} lệnh cách thanh gần nhất quá 5 phút, để NaN thay vì gán bừa")

Không giới hạn: 3,501 lệnh gán được
tolerance=5min: 1,255 lệnh gán được
→ 2,246 lệnh cách thanh gần nhất quá 5 phút, để NaN thay vì gán bừa


### `by=` — ghép asof trong từng mã

Với nhiều mã, `merge_asof` phải biết không được lấy thanh của mã khác.

In [21]:
tick_2ma = pd.concat(
    [
        client.intraday.stock.ticks("HPG", date=NGAY).assign(symbol="HPG"),
        client.intraday.stock.ticks("VCB", date=NGAY).assign(symbol="VCB"),
    ]
).sort_values("time")
thanh_2ma = client.intraday.stock.ohlcv(["HPG", "VCB"], interval="15min", start=NGAY, end=NGAY).sort_values("time")

# ⚠️ Chạy thẳng sẽ HỎNG — và lý do đáng đọc:
try:
    pd.merge_asof(
        tick_2ma,
        thanh_2ma[["time", "symbol", "close"]],
        on="time",
        by="symbol",
        direction="backward",
    )
except pd.errors.MergeError as e:
    print(f"MergeError: {e}")

MergeError: incompatible merge keys [0] <StringDtype(storage='python', na_value=nan)> and <StringDtype(storage='python', na_value=<NA>)>, must be the same type


### ⚠️ Hai kiểu chuỗi của pandas 3.0 vừa cắn

Notebook `51` đo rằng pandas 3.0 có **hai** `StringDtype`: `str`
(thiếu = `np.nan`) là mặc định của pandas, còn `string` (thiếu = `pd.NA`) là thứ
finlens trả về. Đây là lúc khác biệt đó thành một lỗi thật.

`assign(symbol="HPG")` tạo cột bằng **hằng chuỗi của Python**, nên nó ra kiểu
`str`. Cột `symbol` bên frame kia đến từ finlens nên là `string`. `merge_asof`
từ chối ghép hai khoá khác kiểu.

In [22]:
print(f"tick_2ma['symbol']  : {tick_2ma['symbol'].dtype}  na_value={tick_2ma['symbol'].dtype.na_value!r}")
print(f"thanh_2ma['symbol'] : {thanh_2ma['symbol'].dtype}  na_value={thanh_2ma['symbol'].dtype.na_value!r}")
print(f"Bằng nhau? {tick_2ma['symbol'].dtype == thanh_2ma['symbol'].dtype}")

tick_2ma['symbol']  : str  na_value=nan
thanh_2ma['symbol'] : string  na_value=<NA>
Bằng nhau? False


⚠️ **`merge` thường thì KHÔNG báo lỗi này** — nó tự ép kiểu và chạy tiếp. Chỉ
`merge_asof` mới nghiêm ngặt. Nên bạn có thể đã ghép hai kiểu chuỗi khác nhau
nhiều lần mà không biết.

In [23]:
thu_merge = tick_2ma.head(100).merge(thanh_2ma[["symbol", "close"]].head(10), on="symbol", how="left")
print(f"merge thường với hai kiểu khác nhau → chạy được, {len(thu_merge)} dòng")
print(f"  kiểu cột khoá sau ghép: {thu_merge['symbol'].dtype}")

merge thường với hai kiểu khác nhau → chạy được, 500 dòng
  kiểu cột khoá sau ghép: object


**Cách sửa: thống nhất kiểu trước khi ghép.** Ép cả hai về cùng một `dtype` —
ở đây lấy kiểu của frame finlens làm chuẩn.

In [24]:
KIEU_MA = thanh_2ma["symbol"].dtype

theo_ma = pd.merge_asof(
    tick_2ma.astype({"symbol": KIEU_MA}),
    thanh_2ma[["time", "symbol", "close"]]
    .rename(columns={"close": "gia_thanh"})
    .astype({"symbol": KIEU_MA}),
    on="time",
    by="symbol",  # ← không có dòng này là trộn mã
    direction="backward",
)
kiem = theo_ma.dropna(subset=["gia_thanh"])
print(f"Ghép asof có by='symbol': {len(kiem):,} lệnh")
print(f"Kiểm tra không trộn mã — chênh lệch giá lớn nhất giữa lệnh và thanh của nó:")
print(f"  {(kiem['price'] - kiem['gia_thanh']).abs().max():.2f} nghìn VND")
print("  (nếu trộn HPG với VCB thì con số này sẽ hàng chục nghìn VND)")

Ghép asof có by='symbol': 5,466 lệnh
Kiểm tra không trộn mã — chênh lệch giá lớn nhất giữa lệnh và thanh của nó:
  0.50 nghìn VND
  (nếu trộn HPG với VCB thì con số này sẽ hàng chục nghìn VND)


## 7 · Danh sách kiểm trước mỗi lần ghép

Bốn dòng này nên có mặt ở mọi chỗ bạn `merge` trong pipeline sản xuất.

In [25]:
def ghep_an_toan(
    trai: pd.DataFrame,
    phai: pd.DataFrame,
    *,
    on: list[str],
    how: str = "left",
    validate: str = "many_to_one",
) -> pd.DataFrame:
    """Merge có kiểm — ném lỗi thay vì mất dòng hoặc nhân dòng im lặng."""
    truoc = len(trai)
    kq = trai.merge(phai, on=on, how=how, validate=validate)

    if how == "left" and len(kq) != truoc:
        raise ValueError(f"left join đổi số dòng: {truoc:,} → {len(kq):,}")

    cot_moi = [c for c in kq.columns if c not in trai.columns]
    if cot_moi:
        thieu = kq[cot_moi[0]].isna().sum()
        if thieu:
            print(f"⚠️  {thieu:,}/{len(kq):,} dòng ({thieu / len(kq):.1%}) không tra được — cột '{cot_moi[0]}' là NaN")
    return kq


kq = ghep_an_toan(gia, nganh, on=["symbol"])
print(f"✓ {len(kq):,} dòng, {kq.shape[1]} cột")

⚠️  2,420/9,488 dòng (25.5%) không tra được — cột 'icb_name2' là NaN
✓ 9,488 dòng, 9 cột


## Tổng kết

| Bạn cần | Viết |
|---|---|
| Làm giàu bảng sự kiện | `.merge(tra, on=…, how="left", validate="many_to_one")` |
| Bắt bảng tra bị trùng khoá | `validate="many_to_one"` |
| Biết dòng nào từ đâu | `indicator=True` rồi đọc cột `_merge` |
| Xếp chồng nhiều frame | `pd.concat([...], ignore_index=True)` |
| Ghép theo thời gian gần nhất | `pd.merge_asof(..., direction="backward", by="symbol")` |

**Năm điều mang sang notebook sau:**

1. **`how="inner"` là mặc định và nó mất dòng im lặng.** Khi ghép để làm giàu,
   dùng `how="left"` và khẳng định số dòng không đổi.
2. ⚠️ **`validate="many_to_one"` là tham số đáng viết nhất trong pandas.** Nó
   bắt bảng tra bị trùng — lỗi nhân dòng lên mà không ai phát hiện.
3. **Quên một khoá trong `on=` làm nổ số dòng** theo cấp số nhân. `validate=`
   bắt được ngay.
4. **`attrs` mất hay không phụ thuộc dữ liệu**, không phải phép toán: hai frame
   cùng đơn vị thì `concat` giữ, khác đơn vị thì bỏ. Đọc đơn vị trước khi ghép.
5. **`merge_asof(direction="nearest"/"forward")` nhìn vào tương lai.** Trong
   backtest, dùng `"backward"`.
6. ⚠️ **`merge_asof` từ chối ghép hai kiểu chuỗi khác nhau**, còn `merge`
   thường thì tự ép và chạy tiếp. Cột tạo bằng `assign("HPG")` ra kiểu `str`,
   cột từ finlens là `string` — thống nhất `dtype` trước khi ghép.

---

**Tiếp theo:** [`58_time_series.ipynb`](58_time_series.ipynb) — `DatetimeIndex`,
`resample`, `rolling`, và bảng đầy đủ các mã tần suất pandas 3.0 vừa xoá.